# NpuKit — full board smoke

One-shot bring-up on PYNQ-Z2 (same `npukit.bit`):

1. **Matmul** — classic 8×8 + tiled suites  
2. **Glue** — residual / GELU / RMSNorm / Softmax + GEMM tile  
3. **E2E** — synthetic 1-layer transformer block  
4. **ViT** — MNIST tiny-ViT (CPU DS-stem MID=24 + T=16×D=16×MLP32×L=4, `glue=float`, int8 GEMM on FPGA)

**Board run captured 2026-07-27** on `192.168.0.215` with MID=24 `vit_mnist_weights.npz`.

| Suite | Result |
|-------|--------|
| matmul | **12/12 PASS** |
| glue | **ALL BOARD PASS** |
| e2e | **ALL E2E PASS** |
| vit | **ALL VIT PASS** |
| **overall** | **4/4 — ALL SMOKE PASS** |

ViT sample n=64: ref **61/64 (95.3%)**, hw **61/64 (95.3%)**, ref↔hw pred agree **64/64 (100%)**, tensor `max|err|=0` (HW GEMM + A9 float Softmax/RMSNorm/GELU).

Host numpy deploy-quant (full 10k test): **97.98%**.

CLI equivalent:
```bash
sudo bash -lc 'source /etc/profile.d/xrt_setup.sh; source /usr/local/share/pynq-venv/bin/activate; \
  python3 npukit_board_smoke.py /home/xilinx/jupyter_notebooks/npukit.bit --vit-n 64'
```


In [1]:
import importlib
import sys
from pathlib import Path

BIT = "/home/xilinx/jupyter_notebooks/npukit.bit"
HOST = Path("/home/xilinx/jupyter_notebooks")
if not (HOST / "npukit_board_smoke.py").exists():
    HOST = Path("/home/user/fpga/npukit/host")
    BIT = str(HOST.parent / "output" / "npukit.bit")
sys.path.insert(0, str(HOST))

import npukit_board_smoke as smoke

importlib.reload(smoke)
print("bit", BIT, "exists", Path(BIT).exists())
print("host", HOST)

bit /home/xilinx/jupyter_notebooks/npukit.bit exists True
host /home/xilinx/jupyter_notebooks


## Run all suites + summary


In [2]:
VIT_N = 64
results = smoke.run_all(bit_path=BIT, vit_n=VIT_N, matmul_quiet=True)
rc = smoke.print_summary(results)
assert rc == 0, "board smoke failed"

print()
print("ViT detail (from suite log, n=%d):" % VIT_N)
print("  geometry: T=16 D=16 mlp=32 L=4 stem MID=24 glue=float")
vit = next(r for r in results if r.name == "vit")
seen = set()
for line in vit.log.splitlines():
    s = line.strip()
    interesting = (
        s.startswith("ref accuracy")
        or s.startswith("hw  accuracy")
        or s.startswith("ref↔hw")
        or s.startswith("ALL VIT")
        or (s.startswith("tokens:") and "max|err|" in s)
        or (s.startswith("block0.y_out:") and "max|err|" in s)
        or (s.startswith("logits:") and "max|err|" in s)
    )
    if interesting and s not in seen:
        seen.add(s)
        print(" ", s)


NpuKit board smoke summary
  [PASS] matmul    12/12 PASS
  [PASS] glue      ALL BOARD PASS
  [PASS] e2e       ALL E2E PASS
  [PASS] vit       ALL VIT PASS
------------------------------------------------------------
OVERALL: 4/4 suites PASS — ALL SMOKE PASS

ViT detail (from suite log, n=64):
  geometry: T=16 D=16 mlp=32 L=4 stem MID=24 glue=float
  tokens: PASS  max|err|=0  tol=512
  block0.y_out: PASS  max|err|=0  tol=1024
  logits: info  max|err|=0
  ref accuracy on this batch: 61/64 (95.3%)
  hw  accuracy on this batch: 61/64 (95.3%)
  ref↔hw pred agree: 64/64 (100.0%) PASS
  ALL VIT PASS
